# Session 01 — Segments D & E: Built-ins + Standard Library

Six built-in functions and three standard library modules that appear constantly in production AI code. Each one replaces 5+ lines you'd otherwise write by hand.

## Segment D — Key built-in functions

In [ ]:
# getattr(obj, name, default) — get an attribute by its name as a string
# Used when you don't know the attribute name at write time

class APIUsage:
    input_tokens = 1200
    output_tokens = 340
    # cache_read_input_tokens doesn't always exist

usage = APIUsage()

cache_read = getattr(usage, "cache_read_input_tokens", 0)
print(f"cache_read: {cache_read}")  # 0 — attribute doesn't exist, returns default

input_tokens = getattr(usage, "input_tokens", 0)
print(f"input_tokens: {input_tokens}")  # 1200 — attribute exists

`getattr(obj, "attr", default)` is the `.get()` of objects — never crashes on a missing attribute. Used whenever an API response object might or might not have a field depending on the operation.

In [ ]:
# isinstance(obj, Type) — check if an object is an instance of a type

def process(value):
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        return [str(v) for v in value]
    return str(value)

print(process("  hello  "))   # "hello"
print(process([1, 2, 3]))     # ["1", "2", "3"]
print(process(42))            # "42"

In [ ]:
# any() — True if at least one item is truthy
# all() — True if ALL items are truthy
# sum() — sum any iterable
# zip() — pair up two lists element by element

sets = [
    {"weight_kg": 80, "reps": 5},
    {"weight_kg": 0,  "reps": 10},
    {"weight_kg": 82.5, "reps": 3},
]

print(any(s["weight_kg"] > 100 for s in sets))    # False — no set > 100kg
print(any(s["weight_kg"] > 0 for s in sets))       # True — some sets have weight
print(all(s["reps"] > 0 for s in sets))            # True — all sets have reps
print(sum(s["weight_kg"] * s["reps"] for s in sets))  # total volume

In [ ]:
# zip() — pair up two lists
lifts = ["bench", "squat", "deadlift"]
weights = [80, 120, 140]

for lift, weight in zip(lifts, weights):
    print(f"{lift}: {weight}kg")

## Segment E — Standard library: pathlib, datetime, defaultdict

In [ ]:
from pathlib import Path

# Path objects — object-oriented file paths
# The / operator joins paths (works on Windows too, unlike string concatenation)
base = Path("/Users/ganesh/Documents/GaneshFiles/Python_Refresh")
sessions_dir = base / "sessions"
hello_file   = sessions_dir / "session-00" / "hello.py"

print(hello_file)              # full path
print(hello_file.exists())     # True — the file we created earlier
print(hello_file.suffix)       # ".py"
print(hello_file.stem)         # "hello"
print(hello_file.parent)       # sessions/session-00 directory

In [ ]:
# Read a file in one line
content = hello_file.read_text()
print(content)

`Path(__file__).parent` is how production code finds files relative to the current script — used constantly in config loaders and prompt file readers.

In [ ]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)        # always use UTC
print(now.isoformat())                  # ISO format string
print(now.strftime("%Y-%m-%d"))         # "2026-06-05"

# Time arithmetic
four_weeks_ago = now - timedelta(weeks=4)
tomorrow       = now + timedelta(days=1)
print(f"4 weeks ago: {four_weeks_ago.strftime('%Y-%m-%d')}")
print(f"Tomorrow:    {tomorrow.strftime('%Y-%m-%d')}")

# Parse a timestamp string (what APIs return)
dt = datetime.fromisoformat("2026-06-01T09:00:00+00:00")
print(f"Parsed: {dt}")
print(f"Within 4 weeks: {dt >= four_weeks_ago}")

⚡ **Best practice**: Always `datetime.now(timezone.utc)` — never `datetime.now()` without timezone. The bare version returns local machine time, which differs between machines and breaks comparisons across systems.

In [ ]:
from collections import defaultdict

# Regular dict crashes on missing keys:
spend = {}
# spend["user_123"] += 0.003  # KeyError!

# defaultdict provides a default value for any missing key:
spend = defaultdict(float)       # missing keys → 0.0
spend["user_123"] += 0.003
spend["user_123"] += 0.005
spend["user_456"] += 0.001
print(dict(spend))               # {"user_123": 0.008, "user_456": 0.001}

# defaultdict(list) — missing keys → []
by_date = defaultdict(list)
by_date["2026-06-01"].append("workout A")
by_date["2026-06-01"].append("workout B")
by_date["2026-06-03"].append("workout C")
print(dict(by_date))

⚡ **Best practice**: Use `defaultdict` when you're counting or grouping — it eliminates the "check if key exists, initialise, then update" pattern that clutters grouping code.

**Misconception**: `datetime.now()` without `timezone.utc` returns local time — which differs by machine and location. Always use `datetime.now(timezone.utc)` for anything stored in a database or compared across systems.